In [1]:
import pandas as pd
import numpy as np

# Reload the RAW merged data BEFORE imputation if you have it saved,
# otherwise we measure constant-run artifacts as a proxy for imputation depth
df = pd.read_csv('EcoTwin_Final_Dataset.csv', parse_dates=['Timestamp'])
df['Station'] = df['Station'].str.strip()

FEATURES = ['pm2.5','pm10','no2','so2','co','o3','rh','at']
stations = sorted(df['Station'].unique())

def constant_run_fraction(series, min_run=6):
    """Fraction of values inside a run of >= min_run identical consecutive values.
    Long identical runs = imputation filled a gap with a repeated/interpolated value."""
    s = series.values
    if len(s) < min_run:
        return 0.0
    in_run = np.zeros(len(s), dtype=bool)
    i = 0
    while i < len(s):
        j = i + 1
        while j < len(s) and s[j] == s[i]:
            j += 1
        if j - i >= min_run:
            in_run[i:j] = True
        i = j
    return in_run.mean()

print(f"{'Station':<38} " + " ".join(f"{f:>6}" for f in FEATURES))
print("-" * 100)

flagged_stations = []
for station in stations:
    sub = df[df['Station'] == station].sort_values('Timestamp')
    fracs = []
    for feat in FEATURES:
        frac = constant_run_fraction(sub[feat]) * 100
        fracs.append(frac)
    # flag if any feature is >30% synthetic-looking
    max_frac = max(fracs)
    marker = "  <-- CHECK" if max_frac > 30 else ""
    if max_frac > 30:
        flagged_stations.append(station)
    print(f"{station:<38} " + " ".join(f"{f:>6.1f}" for f in fracs) + marker)

print("-" * 100)
print(f"\nStations with >30% constant runs in any feature: {len(flagged_stations)}")
for s in flagged_stations:
    print(f"  {s}")
print("\n(High % = that feature was heavily imputed for that station)")

Station                                 pm2.5   pm10    no2    so2     co     o3     rh     at
----------------------------------------------------------------------------------------------------
Alipur                                    0.2    0.0    0.0    0.0    0.1    0.0    6.8    0.2
Anand Vihar                               0.0    0.0    0.0    0.1    0.4    0.1    0.2    0.0
Ashok Vihar                               0.1    0.0    0.0    0.0    0.5    0.0    2.2    0.0
Aya Nagar                                 0.1    0.0    0.0    0.0    0.5    0.0    0.0    0.0
Bawana                                    0.0    0.0    0.0    0.0    0.2    0.0    4.0    0.0
Burari Crossing                           0.0    0.0    0.0    0.0    0.1    0.2    0.0    0.0
CRRI Mathura Road                         0.1    0.1    0.1    0.0    0.4    0.0    0.0    0.0
Cantonment Area                           0.0    0.0    0.0    0.0    0.0    0.0    0.0    0.0
Chandni Chowk                             4.

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv('EcoTwin_Final_Dataset.csv', parse_dates=['Timestamp'])
df['Station'] = df['Station'].str.strip()

FEATURES = ['pm2.5','pm10','no2','so2','co','o3','rh','at']
stations = sorted(df['Station'].unique())

def smooth_fraction(series, window=6, smooth_thresh=0.01):
    """Fraction of points where local rolling std is near zero (unnaturally smooth).
    Real sensor data jitters; interpolated data is glassy-smooth."""
    s = series.values.astype(float)
    if len(s) < window:
        return 0.0
    # normalise so threshold is scale-independent
    rng = np.nanmax(s) - np.nanmin(s)
    if rng == 0:
        return 1.0
    s_norm = (s - np.nanmin(s)) / rng
    roll_std = pd.Series(s_norm).rolling(window, center=True).std()
    return (roll_std < smooth_thresh).mean()

print(f"{'Station':<38} " + " ".join(f"{f:>6}" for f in FEATURES))
print("-" * 100)
for station in stations:
    sub = df[df['Station'] == station].sort_values('Timestamp')
    fracs = [smooth_fraction(sub[f]) * 100 for f in FEATURES]
    marker = "  <-- CHECK" if max(fracs) > 25 else ""
    print(f"{station:<38} " + " ".join(f"{f:>6.1f}" for f in fracs) + marker)
print("\n(High % = long unnaturally-smooth stretches = likely interpolated)")

Station                                 pm2.5   pm10    no2    so2     co     o3     rh     at
----------------------------------------------------------------------------------------------------
Alipur                                   32.5   10.6   12.2   43.6   33.2   34.2    7.0   18.5  <-- CHECK
Anand Vihar                              16.6    3.6   19.4   34.6   18.8   54.3    5.0   22.2  <-- CHECK
Ashok Vihar                              29.6    9.9   12.9   44.2   17.8   28.1    5.4   22.6  <-- CHECK
Aya Nagar                                31.0   10.7   39.9   37.2   37.2   27.7    6.2   17.9  <-- CHECK
Bawana                                   26.5    8.9   43.1   58.2   32.9   29.5    5.7   19.4  <-- CHECK
Burari Crossing                          24.6   10.8   61.0   37.2   91.4   31.6    6.2   17.9  <-- CHECK
CRRI Mathura Road                        35.2    8.2   51.1   37.2   27.6   17.1    6.2   17.9  <-- CHECK
Cantonment Area                          48.0   20.3   45.1   

In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv('EcoTwin_Final_Dataset.csv', parse_dates=['Timestamp'])
df['Station'] = df['Station'].str.strip()

FEATURES = ['pm2.5','pm10','no2','so2','co','o3','rh','at']
stations = sorted(df['Station'].unique())

print(f"{'Station':<38} {'% rows matching another station (any feat)':>45}")
print("-" * 85)

for feat in ['pm2.5']:  # focus on the target; change to loop over FEATURES if wanted
    print(f"\n=== Feature: {feat} ===")
    wide = df.pivot_table(index='Timestamp', columns='Station', values=feat)
    for station in stations:
        if station not in wide.columns:
            continue
        col = wide[station]
        # count timestamps where this station's value exactly equals another station's
        others = wide.drop(columns=[station])
        exact_match = (others.sub(col, axis=0).abs() < 1e-6).any(axis=1)
        pct = exact_match.mean() * 100
        marker = "  <-- borrowed" if pct > 15 else ""
        print(f"  {station:<38} {pct:>6.1f}%{marker}")
print("\n(High % = this station's values were copied from a neighbour = spatial fill)")

Station                                   % rows matching another station (any feat)
-------------------------------------------------------------------------------------

=== Feature: pm2.5 ===
  Alipur                                   27.6%  <-- borrowed
  Anand Vihar                              16.8%  <-- borrowed
  Ashok Vihar                              30.9%  <-- borrowed
  Aya Nagar                                 0.6%
  Bawana                                   19.2%  <-- borrowed
  Burari Crossing                           0.6%
  CRRI Mathura Road                         0.5%
  Cantonment Area                           0.1%
  Chandni Chowk                             2.5%
  Commonwealth Sports Complex               0.2%
  DTU                                       7.0%
  Dr. Karni Singh Shooting Range           27.5%  <-- borrowed
  Dwarka-Sector 8                          31.3%  <-- borrowed
  IGI Airport (T3)                          0.5%
  IGNOU_Maidan Garhi               

In [4]:
import pandas as pd
import numpy as np

df = pd.read_csv('EcoTwin_Final_Dataset.csv', parse_dates=['Timestamp'])
df['Station'] = df['Station'].str.strip()

stations = sorted(df['Station'].unique())
t_min, t_max = df['Timestamp'].min(), df['Timestamp'].max()
full_range = pd.date_range(t_min, t_max, freq='1h')
expected = len(full_range)

print(f"Full expected hourly range: {t_min} -> {t_max}")
print(f"Expected hours per station: {expected:,}\n")
print(f"{'Station':<38} {'Rows':>7} {'Dupes':>6} {'Missing':>8} {'Coverage':>9}")
print("-" * 75)

for station in stations:
    sub = df[df['Station'] == station]
    n_rows = len(sub)
    n_dupes = sub['Timestamp'].duplicated().sum()
    unique_ts = sub['Timestamp'].nunique()
    missing = expected - unique_ts
    coverage = unique_ts / expected * 100
    marker = "  <-- gaps" if coverage < 98 else ""
    print(f"{station:<38} {n_rows:>7,} {n_dupes:>6} {missing:>8,} {coverage:>8.1f}%{marker}")

print("\n(Missing = hours that were fabricated by resampling if coverage < 100%)")

Full expected hourly range: 2024-01-01 00:00:00 -> 2026-04-10 00:00:00
Expected hours per station: 19,921

Station                                   Rows  Dupes  Missing  Coverage
---------------------------------------------------------------------------
Alipur                                  19,921      0        0    100.0%
Anand Vihar                             19,921      0        0    100.0%
Ashok Vihar                             19,921      0        0    100.0%
Aya Nagar                               19,921      0        0    100.0%
Bawana                                  19,921      0        0    100.0%
Burari Crossing                         19,921      0        0    100.0%
CRRI Mathura Road                       19,921      0        0    100.0%
Cantonment Area                         19,921      0        0    100.0%
Chandni Chowk                           19,921      0        0    100.0%
Commonwealth Sports Complex             19,921      0        0    100.0%
DTU           

In [5]:
import pandas as pd
import numpy as np
import glob
import os

# Updated to new project location
RAW_PATH = r'D:\FYP\Eco-Twin\Raw Sets'

COL_MAP = {
    'PM2.5': 'pm2.5', 'PM10': 'pm10', 'NO2': 'no2', 'SO2': 'so2',
    'CO': 'co', 'Ozone': 'o3', 'O3': 'o3', 'RH': 'rh',
    'WS': 'ws', 'WD': 'wd', 'AT': 'at', 'Temp': 'at',
    'Ambient Temperature': 'at',
}
NULL_STRINGS = ['None', 'N/A', '-', '--', ' ', 'null', 'NA', 'nan', 'NaN']
FEATURES = ['pm2.5', 'pm10', 'no2', 'so2', 'co', 'o3', 'rh', 'ws', 'wd', 'at']

all_files = glob.glob(os.path.join(RAW_PATH, '*.xls*'))
print(f"Found {len(all_files)} raw files\n")

print(f"{'Station':<38} {'Rows':>7} " + " ".join(f"{f:>6}" for f in FEATURES))
print("-" * 115)

summary = []
for filepath in sorted(all_files):
    station = os.path.basename(filepath).rsplit('.', 1)[0].strip()
    try:
        engine = 'xlrd' if filepath.endswith('.xls') else 'openpyxl'
        df = pd.read_excel(filepath, engine=engine)
        df.replace(NULL_STRINGS, np.nan, inplace=True)
        df.rename(columns=COL_MAP, inplace=True)

        n_rows = len(df)
        missing_pct = {}
        for feat in FEATURES:
            if feat in df.columns:
                vals = pd.to_numeric(df[feat], errors='coerce')
                missing_pct[feat] = vals.isna().mean() * 100
            else:
                missing_pct[feat] = 100.0

        row = {'Station': station, 'Rows': n_rows, **missing_pct}
        summary.append(row)

        cells = [f"{missing_pct[feat]:>6.1f}" for feat in FEATURES]
        print(f"{station:<38} {n_rows:>7,} " + " ".join(cells))

    except Exception as e:
        print(f"{station:<38} ERROR: {e}")

summary_df = pd.DataFrame(summary)

print("\n" + "=" * 60)
print("  FLAGS — stations with problematic PM2.5 (the target)")
print("=" * 60)
for _, r in summary_df.sort_values('pm2.5', ascending=False).iterrows():
    pm = r['pm2.5']
    if pm > 50:
        verdict = "DROP — mostly missing"
    elif pm > 25:
        verdict = "CAUTION — heavily imputed"
    else:
        verdict = "OK"
    if pm > 25:
        print(f"  {r['Station']:<38} PM2.5 missing: {pm:>5.1f}%   {verdict}")

print("\n" + "=" * 60)
print("  FLAGS — entirely missing parameters (100% absent)")
print("=" * 60)
for _, r in summary_df.iterrows():
    dead = [f for f in FEATURES if r[f] >= 99.9]
    if dead:
        print(f"  {r['Station']:<38} missing entirely: {dead}")

summary_df.to_csv('raw_station_missingness.csv', index=False)
print(f"\nSaved full report to raw_station_missingness.csv")

Found 45 raw files

Station                                   Rows  pm2.5   pm10    no2    so2     co     o3     rh     ws     wd     at
-------------------------------------------------------------------------------------------------------------------
Alipur                                  19,921    2.7    2.9    1.9    2.6    4.3    3.7    1.6  100.0    1.6    1.6
Anand Vihar                             19,921   11.4   11.3    9.0   14.6    9.2   11.4    8.0    8.8    8.3    8.0
Ashok Vihar                             19,921    1.9    1.7    0.8    3.3    5.0    1.1    0.7    0.7    0.7    0.7
Aya Nagar                               19,921    5.0    4.0    4.0  100.0    3.0    4.0  100.0  100.0  100.0  100.0
Bawana                                  19,921    2.2    1.8    1.0    1.4    1.3    1.1    0.4    4.7    1.8    0.4
Burari Crossing                         19,921    4.8    4.4    4.4  100.0    4.4    3.8  100.0  100.0  100.0  100.0
CRRI Mathura Road                       19,92